In [1]:
#  DATA SOURCE
# https://www.kaggle.com/saurabh00007/diabetescsv/data


In [1]:
# Import required libraries
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import sklearn

from sklearn.neural_network import MLPRegressor

# Import necessary modules
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from math import sqrt
from sklearn.metrics import r2_score


from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
# from xgboost import XGBClassifier
# from sklearn.metrics import plot_roc_curve

from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

from scipy import interp
import numpy as np
from sklearn.metrics import auc
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

from sklearn.metrics import classification_report,confusion_matrix


from sklearn.preprocessing import StandardScaler

import pandas as pd
from sklearn.preprocessing import label_binarize

In [2]:
fName='GallStone_dataset-uci.csv'
df = pd.read_csv(fName) 
print(df.shape)
df.describe().transpose()

(319, 39)


,count,mean,std,min,25%,50%,75%,max
Gallstone Status,319.0,0.495298,0.500763,0.00,0.000,0.000000,1.000,1.00
Age,319.0,48.068966,12.114558,20.00,38.500,49.000000,56.000,96.00
Gender,319.0,0.492163,0.500724,0.00,0.000,0.000000,1.000,1.00
Comorbidity,319.0,0.335423,0.517340,0.00,0.000,0.000000,1.000,3.00
Coronary Artery Disease (CAD),319.0,0.037618,0.190568,0.00,0.000,0.000000,0.000,1.00
Hypothyroidism,319.0,0.028213,0.165841,0.00,0.000,0.000000,0.000,1.00
Hyperlipidemia,319.0,0.025078,0.156609,0.00,0.000,0.000000,0.000,1.00
Diabetes Mellitus (DM),319.0,0.134796,0.342042,0.00,0.000,0.000000,0.000,1.00
Height,319.0,167.156740,10.053030,145.00,159.500,168.000000,175.000,191.00
Weight,319.0,80.564890,15.709069,42.90,69.600,78.800000,91.250,143.50


In [3]:
target_column = ['Gallstone Status'] 
predictors = list(set(list(df.columns))-set(target_column))
df[predictors] = df[predictors]/df[predictors].max()
df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
Gallstone Status,319.0,0.495298,0.500763,0.000000,0.000000,0.000000,1.000000,1.0
Age,319.0,0.500718,0.126193,0.208333,0.401042,0.510417,0.583333,1.0
Gender,319.0,0.492163,0.500724,0.000000,0.000000,0.000000,1.000000,1.0
Comorbidity,319.0,0.111808,0.172447,0.000000,0.000000,0.000000,0.333333,1.0
Coronary Artery Disease (CAD),319.0,0.037618,0.190568,0.000000,0.000000,0.000000,0.000000,1.0
Hypothyroidism,319.0,0.028213,0.165841,0.000000,0.000000,0.000000,0.000000,1.0
Hyperlipidemia,319.0,0.025078,0.156609,0.000000,0.000000,0.000000,0.000000,1.0
Diabetes Mellitus (DM),319.0,0.134796,0.342042,0.000000,0.000000,0.000000,0.000000,1.0
Height,319.0,0.875166,0.052634,0.759162,0.835079,0.879581,0.916230,1.0
Weight,319.0,0.561428,0.109471,0.298955,0.485017,0.549129,0.635889,1.0


In [4]:
X = df[predictors]
Y = df[target_column]


scale = StandardScaler()
scaledX = scale.fit_transform(X)


In [5]:
X_train, X_test, Y_train, Y_test = train_test_split(scaledX, Y, test_size=0.10, random_state=40)
print(X_train.shape); 
print(X_test.shape)

(287, 38)
(32, 38)


In [6]:

def withoutCV(model, X_test, Y_test):

    # print('\n ******* Without CROSS-VALIDATION ******** \n')

    y_pred = model.predict( X_test)
    tn, fp, fn, tp = confusion_matrix(Y_test, y_pred).ravel()

    # conf_mat = [[ tp,  fn],
    #             [fp,  tn]]
    conf_mat = [[tn, fp],
                [fn, tp]]

    # print(conf_mat)
    acc = (tp+tn)/(tp+tn+fp+fn)
    # print("Accuracy: %0.4f" % (acc))
    sens = (tp)/(tp+fn)
    # print("Sensitivity/Recall: %0.4f" % (sens))
    spec = (tn)/(tn+fp)
    # print("Specificity: %0.4f" % (spec))
    prec = (tp)/(tp+fp)
    # print("prec: %0.4f" % (prec))
    result = " \t Sn: " + str(round(sens, 3)) + " \t Sp: " + str(round(spec,3)) + " \t Acc: " + str(round(acc,3)) + " \t Pr: " + str(round(prec,3))   
    return result
    

    # print(confusion_matrix(Y_test,y_pred))
    # print(classification_report(Y_test,y_pred))

In [7]:

def withCV(model, X_train, X_test, Y_train, Y_test):
    # print('\n ********** using CROSS-VALIDATION  ********** \n')
    y_pred = cross_val_predict(model, X, Y, cv=10)
    tn, fp, fn, tp = confusion_matrix(Y, y_pred).ravel()

    # conf_mat = [[ tp,  fn],
    #             [fp,  tn]]
    conf_mat = [[tn, fp],
                [fn, tp]]

    # print(conf_mat)
    acc = (tp+tn)/(tp+tn+fp+fn)
    # print("Accuracy: %0.4f" % (acc))
    sens = (tp)/(tp+fn)
    # print("Sensitivity/Recall: %0.4f" % (sens))
    spec = (tn)/(tn+fp)
    # print("Specificity: %0.4f" % (spec))
    prec = (tp)/(tp+fp)
    # print("prec: %0.4f" % (prec))
    result = " \t Sn: " + str(round(sens, 3)) + " \t Sp: " + str(round(spec,3)) + " \t Acc: " + str(round(acc,3)) + " \t Pr: " + str(round(prec,3))   
    return result
    
    # print(confusion_matrix(Y,y_pred))
    # print(classification_report(Y,y_pred))

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import GradientBoostingClassifier



# 2. Define model list
models = [
    ("DecisionTree", DecisionTreeClassifier() ), 
    ("LogisticRegression", LogisticRegression(random_state=0)),
    ("RandomForest", RandomForestClassifier(n_estimators=100) ),
    ("GradientBoostingClassifier ", GradientBoostingClassifier(n_estimators=100, learning_rate=1.0, max_depth=1, random_state=0)), 
    ("ExtraTreeClassify" , ExtraTreesClassifier(n_estimators=10, max_depth=None,min_samples_split=2, random_state=0) ),
    ("SVMlinear" , SVC(kernel='linear', C=1.0, random_state=1) ), 
    ("SVMrbf" , SVC(kernel='rbf', random_state=1, gamma=0.10, C=10.0) ), 
    ("NB" , GaussianNB()), 
    
]



In [9]:
# 3. Train in a loop for WITHOUT CV
result= "";
 
# 3. Train in a loop without CV
result= "";
for name, model in models:
    print("\n  ****** Training FOR:    ", name , "************")
    model = model.fit(X_train, Y_train)
    result = result +  name + ": " + withoutCV(model, X_test, Y_test ) + " \n"
     
    
print(result)


  ****** Training FOR:     DecisionTree ************

  ****** Training FOR:     LogisticRegression ************

  ****** Training FOR:     RandomForest ************

  ****** Training FOR:     GradientBoostingClassifier  ************


/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/utils/validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:8: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/ensemble/_gb.py:494: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)



  ****** Training FOR:     ExtraTreeClassify ************

  ****** Training FOR:     SVMlinear ************

  ****** Training FOR:     SVMrbf ************

  ****** Training FOR:     NB ************
DecisionTree:  	 Sn: 0.632 	 Sp: 0.692 	 Acc: 0.656 	 Pr: 0.75 
LogisticRegression:  	 Sn: 0.737 	 Sp: 0.923 	 Acc: 0.812 	 Pr: 0.933 
RandomForest:  	 Sn: 0.737 	 Sp: 0.923 	 Acc: 0.812 	 Pr: 0.933 
GradientBoostingClassifier :  	 Sn: 0.737 	 Sp: 0.769 	 Acc: 0.75 	 Pr: 0.824 
ExtraTreeClassify:  	 Sn: 0.579 	 Sp: 0.846 	 Acc: 0.688 	 Pr: 0.846 
SVMlinear:  	 Sn: 0.737 	 Sp: 0.923 	 Acc: 0.812 	 Pr: 0.933 
SVMrbf:  	 Sn: 0.632 	 Sp: 0.769 	 Acc: 0.688 	 Pr: 0.8 
NB:  	 Sn: 0.105 	 Sp: 1.0 	 Acc: 0.469 	 Pr: 1.0 



/Users/tanviralam/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:8: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/utils/validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/utils/validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/utils/validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n

In [10]:
# 3. Train in a loop for WITH CV
result= "";
for name, model in models:
    print("\n  ****** Training FOR:    ", name , "************")
    y_score = model.fit(X_train, Y_train)
    result = result +  name + ": " + withCV(model, X_train, X_test, Y_train, Y_test ) + " \n"
    

    
print(result)


  ****** Training FOR:     DecisionTree ************

  ****** Training FOR:     LogisticRegression ************


/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/utils/validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/utils/validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/utils/validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/utils/validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expec


  ****** Training FOR:     RandomForest ************


/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/model_selection/_validation.py:1044: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/model_selection/_validation.py:1044: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/model_selection/_validation.py:1044: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/model_selection/_validat


  ****** Training FOR:     GradientBoostingClassifier  ************


/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/ensemble/_gb.py:494: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/ensemble/_gb.py:494: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/ensemble/_gb.py:494: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/ensemble/_gb.py:494: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please chan


  ****** Training FOR:     ExtraTreeClassify ************

  ****** Training FOR:     SVMlinear ************


/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/model_selection/_validation.py:1044: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/model_selection/_validation.py:1044: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/model_selection/_validation.py:1044: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  estimator.fit(X_train, y_train, **fit_params)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/model_selection/_validat


  ****** Training FOR:     SVMrbf ************

  ****** Training FOR:     NB ************
DecisionTree:  	 Sn: 0.658 	 Sp: 0.677 	 Acc: 0.668 	 Pr: 0.667 
LogisticRegression:  	 Sn: 0.684 	 Sp: 0.739 	 Acc: 0.712 	 Pr: 0.72 
RandomForest:  	 Sn: 0.722 	 Sp: 0.758 	 Acc: 0.74 	 Pr: 0.745 
GradientBoostingClassifier :  	 Sn: 0.722 	 Sp: 0.696 	 Acc: 0.708 	 Pr: 0.699 
ExtraTreeClassify:  	 Sn: 0.627 	 Sp: 0.764 	 Acc: 0.696 	 Pr: 0.723 
SVMlinear:  	 Sn: 0.665 	 Sp: 0.764 	 Acc: 0.715 	 Pr: 0.734 
SVMrbf:  	 Sn: 0.665 	 Sp: 0.758 	 Acc: 0.712 	 Pr: 0.729 
NB:  	 Sn: 0.127 	 Sp: 0.963 	 Acc: 0.549 	 Pr: 0.769 



/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/utils/validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/utils/validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/utils/validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/utils/validation.py:993: DataConversionWarning: A column-vector y was passed when a 1d array was expec

In [11]:

from sklearn.ensemble import StackingClassifier

n = "StackingClassifier"
estimators = [

    ('rf', RandomForestClassifier(n_estimators=10, random_state=42)) ,
    ('mylr' , LogisticRegression(random_state=0) ), 
    ('svrlinear', SVC(kernel='linear', C=1.0, random_state=1) ), 
    ('svrbf', SVC(kernel='rbf', random_state=1, gamma=0.10, C=10.0) ), 
#     ("GradientBoostingClassifier ", GradientBoostingClassifier(n_estimators=100, learning_rate=1.0, max_depth=1, random_state=0)), 
#     ("ExtraTreeClassify" , ExtraTreesClassifier(n_estimators=10, max_depth=None,min_samples_split=2, random_state=0) ),
    
    
]

model = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression() )
print(n)

y_score = model.fit(X_train,Y_train)
#scores = cross_val_score(c, X_train, Y_train, cv=5, scoring='accuracy')
#scores = cross_val_score(c, X_train, Y_train, cv=5, scoring='balanced_accuracy')
result = "Stacking with CV" + ": " + withCV(model, X_train, X_test, Y_train, Y_test ) + " \n"
print(result)   

result = "Stacking -withoutCV" + ": " + withoutCV(model, X_test, Y_test ) + " \n"
print(result)    

# withoutCV(model, X_train, X_test, Y_train, Y_test )

StackingClassifier


/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/preprocessing/_label.py:98: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/preprocessing/_label.py:133: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/preprocessing/_label.py:98: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Users/tanviralam/anaconda3/lib/python3.7/site-packages/sklearn/preprocessing/_label.py:133: DataConversionWarning: A column-vector y was passed when a 1d a

Stacking with CV:  	 Sn: 0.69 	 Sp: 0.795 	 Acc: 0.743 	 Pr: 0.768 

Stacking -withoutCV:  	 Sn: 0.737 	 Sp: 0.923 	 Acc: 0.812 	 Pr: 0.933 

